In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from transformers import AutoTokenizer

from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

EPOCHS = 2
LEARNING_RATE = 0.01
BATCH_SIZE = 64

DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA device found.", torch.cuda.is_available())
    EPOCHS = 7
    BATCH_SIZE = 128
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print ("MPS device found.", torch.backends.mps.is_available())
    EPOCHS = 3
else:
    print("Using CPU device.")
    #BATCH_SIZE = 64

CUDA device found. True


In [2]:
from pathlib import Path
from zipfile import ZipFile

url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"

data_dir = Path("data/text_classify")
if not data_dir.exists():
    data_dir.mkdir(exist_ok=True, parents=True)

train_file_path = ""
test_file_path = ""
for url in (url_1, url_2):
    zip_path = data_dir / Path(url).name
    if not zip_path.exists():
        torch.hub.download_url_to_file(url, str(zip_path))
    with ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)
    if "train" in url:
        train_file_path = data_dir / "train.csv"
    else:        
        test_file_path = data_dir / "test.csv"

train_df = pd.read_csv(train_file_path)
test_df = pd.read_csv(test_file_path)
train_df.head()

100%|██████████| 26.3M/26.3M [00:01<00:00, 19.2MB/s]
100%|██████████| 22.0M/22.0M [00:01<00:00, 16.1MB/s]


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
test_df.head()

,id,comment_text
0,1,Yo bitch Ja Rule is more succesful then you'll...
1,2,== From RfC == \n\n The title is fine as it is...
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,4,":If you have a look back at the source, the in..."
4,5,I don't anonymously edit articles at all.


### Create a Dataset

For the dataset, we need to manage how things are put together, there's not one specific answer, we just need to match up how we treat the data here with how we use it later. Some decisions I made are:
<ul>
<li> Take in the entire dataset, with column names to indicate what to use. </li>
<li> Return several things packaged together, the text, the labels, and the length of the text. </li>
<li> Tokenize everything on creation. </li>
</ul>

### Packed and Padded Sequences

When working with natural language, we unfortunately rarely get every document to be the identical length, post tokenization. Our models expect everything to be an exact length, so what do we do?

One answer is to pad everything to the same length, or to insert a special token to fill up all the empty space. PyTorch offers an even better solution, which is to pack the sequences. A packed sequence is a special data structure that packages the sequences together with their lengths, allowing the model to ignore the padding and focus on the actual data. This can lead to better performance and faster training times. This isn't conceptually different, it is mainly a shortcut to make our lives easier.

#### Collate

This dataset adds a collate function. This function serves to 'package' the data for the dataloader in a way other than the default of getitem. For this example, it allows us to do the sequence padding. 

The collate_fn function that is added to the dataloader constructor tells the dataloader to use that function to get each batch of data. The function will take a normal batch and pad it, efficiently. The sequences are padded out only to the length needed by the batch, not the same overall value for all. This saves processing, and can add up in a large model. 

In [4]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, df, text_col, label_cols=None, max_length=128):
        self.df = df
        self.text_col = text_col
        self.label_cols = label_cols
        self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        self.vocab_size = len(self.tokenizer.get_vocab())
        self.max_length = max_length
        self.tokenizeText()

    def tokenizeText(self):
        # Tokenize WITHOUT padding — only truncate to max_length
        self.df["tokenized"] = self.df[self.text_col].apply(
            lambda x: self.tokenizer(x, truncation=True, padding=False, max_length=self.max_length)
        )

    def getVocabSize(self):
        return self.vocab_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = self.df.iloc[idx]
        text = item[self.text_col]
        tokenized = item["tokenized"]

        input_ids = torch.tensor(tokenized["input_ids"], dtype=torch.long)
        length = len(tokenized["input_ids"])

        result = {
            "text": text,
            "input_ids": input_ids,
            "length": length,
        }

        if self.label_cols is not None:
            labels = item[self.label_cols].values.astype(np.float32)
            result["labels"] = torch.tensor(labels, dtype=torch.float32)

        return result


def collate_fn(batch):
    """Pad sequences to the longest in the batch and return lengths for packing."""
    input_ids = [item["input_ids"] for item in batch]
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    texts = [item["text"] for item in batch]

    # Pad to the max length within this batch (pad value = 0)
    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=0)

    result = {
        "text": texts,
        "input_ids": input_ids_padded,
        "lengths": lengths,
    }

    if "labels" in batch[0]:
        labels = torch.stack([item["labels"] for item in batch])
        result["labels"] = labels

    return result


train_dataset = MultiLabelClassificationDataset(train_df, text_col="comment_text", label_cols=["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"])
vocab_size = train_dataset.getVocabSize()
print(f"Vocabulary size: {vocab_size}")
# random split for validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

prediction_dataset = MultiLabelClassificationDataset(test_df, text_col="comment_text")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
prediction_loader = DataLoader(prediction_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 30522


## Create a Model

The model here is pretty standard: 

- The first layer does embedding, providing the vector representation of the text.
- The second layer is the LSTM.
- The final layer is a linear layer to get the output into the right shape.

### Multilabel Classification

This task is a multilabel classification, or assigning an item into 0 or more groups, where they can be in more than one. This requires changing something like a parameter, if the model can handle it, for classical models. For neural networks, we can create it by manipulating the shape of the prediction and the loss calculation. 

![Multilablel](images/multilabel.jpg "Multilabel")

The output of the model is a vector of probabilities for each class, which is more or less like a set of independent binary classifications. We setup the model to have 6 output nodes, and we train the model with each of those nodes being 1/0 from the label. Remember - the model will learn whatever we set as a target. 

As a side note, this is a small example of the flexibility of the neural network as a predictive tool. If we are able to provide any output(s) along with any set of predictive features, we can get a pretty smart predictive model, and we *don't* have to specify how to make the prediction from the features, that part is magic. With fast GPUs and large datasets, this is how models are able to learn to do many different things well(-ish) - a massive amount of video/sensor readings, server rentals, and "was that a crash" labels can allow us to teach a car to drive while knowing very little of how we did it. 

### Output and Loss

The loss function we use is `BCEWithLogitsLoss`, which combines a sigmoid layer and the binary cross-entropy loss in one single class. This loss expects logits as an input, so we don't need an activation layer here. Each individual prediction will be one sixth of the total loss, which works fine here. I looked this up by searching "pytorch loss function multilabel classification" and reading an example; this is probably good practice in general, there are often more than one way that will work, such as if activation is a manual layer or built into the loss function, and sometimes one way is preferable for technical reasons behind the scenes. 

### The State of Concatenate

<b>I was (maybe) wrong</b> earlier, please hold your shock. I read, in separate texts, that the LSTM layer will auto-concatenate when using bidirectional layers, and also that we need to do it manually. I'll play it safe here and do it manually, this is the 'safe' choice as there's no doubt. The h_forward and h_backward variables below are the hidden states of the last forward and backward layers, and I am concatenating them together to be sent through the output.

I looked up the indices and dimensions to grab and combine the layers from an example, as this is a common problem. If we were doing something odd and were not sure how to manipulate the shapes, the best bet is to try to draw out one (or part of one) example record of each, figure out the correct operations, then use print statements to make sure that one record is manipulated correctly before passing all of the data. 

In [ ]:
class LSTM_classifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, dropout=0.4, num_layers=2):
        super(LSTM_classifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, num_layers=num_layers, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # * 2 for bidirectional

    def forward(self, input_ids, lengths=None):
        embedded = self.embedding(input_ids)

        # If lengths provided, pack the padded sequence so the LSTM ignores padding
        if lengths is not None:
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, (h_n, c_n) = self.lstm(packed)
        else:
            lstm_out, (h_n, c_n) = self.lstm(embedded)

        # h_n shape: (num_layers * num_directions, batch, hidden_size)
        # For a bidirectional LSTM the last layer has two hidden states: forward and backward.
        # Take the final forward and backward hidden states (last layer) and concatenate them.
        # forward hidden state is at -2, backward at -1
        h_forward = h_n[-2]
        h_backward = h_n[-1]
        h_cat = torch.cat((h_forward, h_backward), dim=1)

        out = self.fc(h_cat)
        #final_out, 
        return out
    
    def predictions(self, inputs, as_dataframe=False, column_names=["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]):
        self.eval()
        all_preds = []
        with torch.no_grad():
            for batch in inputs:
                data_ = batch["input_ids"].to(DEVICE)
                lengths = batch["lengths"]
                outputs = self.forward(data_, lengths)
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                for i in range(preds.shape[0]):
                    text = batch["text"][i]
                    pred_vector = preds[i]
                    all_preds.append(pred_vector)
                #all_preds.append(preds)
        all_preds = np.vstack(all_preds)
        if as_dataframe:
            return pd.DataFrame(all_preds, columns=column_names)
        return all_preds

### Training Time

The training loop is pretty basic, the only unique parts here are:
<ul>
<li> Each batch is a dictionary, so we need to pull out the relevant parts. </li>
<li> The output accuracy requires a bit more calculating. </li>
</ul>

In [6]:
def train_loop(model, train_loader, optimizer, loss_fn, val_loader, epochs=20, classes=["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]):
    model.to(DEVICE)
    train_accuracies = {class_name: [] for class_name in classes}
    val_accuracies = {class_name: [] for class_name in classes}
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        # Per-epoch accumulators
        epoch_train_acc = {class_name: [] for class_name in classes}
        epoch_val_acc = {class_name: [] for class_name in classes}

        with torch.set_grad_enabled(True):
            for batch in train_loader:
                data_ = batch["input_ids"].to(DEVICE)
                lengths = batch["lengths"]
                labels = batch["labels"].to(DEVICE)
                optimizer.zero_grad()
                outputs = model(data_, lengths)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                # calcing accuracy for each class
                with torch.no_grad():
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                    for i, class_name in enumerate(classes):
                        correct = (preds[:, i] == labels[:, i]).sum().item()
                        total = labels.size(0)
                        acc = correct / total
                        epoch_train_acc[class_name].append(acc)

        # Evaluate on validation set
        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                data_ = batch["input_ids"].to(DEVICE)
                lengths = batch["lengths"]
                labels = batch["labels"].to(DEVICE)
                outputs = model(data_, lengths)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                for i, class_name in enumerate(classes):
                    correct = (preds[:, i] == labels[:, i]).sum().item()
                    total = labels.size(0)
                    acc = correct / total
                    epoch_val_acc[class_name].append(acc)

        # Store epoch-level mean accuracy per class
        for class_name in classes:
            train_accuracies[class_name].append(np.mean(epoch_train_acc[class_name]))
            val_accuracies[class_name].append(np.mean(epoch_val_acc[class_name]))

        epoch_val_mean = np.mean([val_accuracies[c][-1] for c in classes])
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f} Val Acc: {epoch_val_mean:.4f}")
    return train_accuracies, val_accuracies

In [ ]:

lstm_model = LSTM_classifier(vocab_size=vocab_size, embedding_dim=128, hidden_dim=64, output_dim=6)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)

train_accuracies, val_accuracies = train_loop(lstm_model, train_loader, optimizer, loss_fn, val_loader, epochs=EPOCHS)

Epoch 1/7, Loss: 0.0691 Val Acc: 0.9812
Epoch 2/7, Loss: 0.0479 Val Acc: 0.9819


### Generate "Real" Predictions

The test data here isn't really "test", it is actual stuff with no answers. We need to detect the spam!!!!!!!!!!

In [ ]:
'''
class answerVectorizer:
    def __init__(self, classes=["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]):
        self.classes = classes
        self.class_to_index = {class_name: idx for idx, class_name in enumerate(classes)}
    
    def vectorize(self, answer_dict):
        vector = np.zeros(len(self.classes), dtype=np.float32)
        for class_name, value in answer_dict.items():
            if class_name in self.class_to_index:
                idx = self.class_to_index[class_name]
                vector[idx] = 1.0 if value else 0.0
        return vector
    
    def devectorize(self, vector):
        answer_dict = {}
        for idx, class_name in enumerate(self.classes):
            answer_dict[class_name] = bool(vector[idx])
        return answer_dict
'''

In [ ]:
pred_df = lstm_model.predictions(prediction_loader, as_dataframe=True)
#pred_df.head()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Run predictions and collect into a DataFrame
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

lstm_model.to(DEVICE)
lstm_model.eval()
rows = []
with torch.no_grad():
    for batch in prediction_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        lengths = batch["lengths"]
        outputs = lstm_model(input_ids, lengths)

        probs = torch.sigmoid(outputs).cpu().numpy()
        texts = batch["text"]
        for i, text in enumerate(texts):
            row = {"text": text}
            for j, cname in enumerate(classes):
                row[cname] = float(probs[i, j])
            rows.append(row)

pred_df = pd.DataFrame(rows)
pred_df["average_offensiveness"] = pred_df[classes].mean(axis=1).round(3)
pred_df[classes] = pred_df[classes].round(3)

pred_df.sort_values("average_offensiveness", ascending=False).head(20)

,text,toxic,severe_toxic,obscene,threat,insult,identity_hate,average_offensiveness
81553,abdi is a lil bitch ass nigga,1.000,0.775,0.992,0.218,0.963,0.544,0.749
100986,PATHETIC BITCH ASS SAND NIGGER SNITCH FUCK U S...,1.000,0.795,0.993,0.202,0.962,0.511,0.744
125610,== GAY GAY GAY == \n\n GAY FUCKING GAY FUCKING...,1.000,0.711,0.990,0.288,0.962,0.501,0.742
74354,== GAY == \n\n YOUR A WANKEST WANKEST WANKEST ...,1.000,0.729,0.991,0.277,0.957,0.491,0.741
129783,No Because Tom Is a MotherTrucking Bitch Ass N...,0.999,0.696,0.988,0.302,0.944,0.519,0.741
74865,id fuck da bitch fo sho my nigga,1.000,0.762,0.992,0.212,0.952,0.527,0.741
149396,I Fucking hate niggers they are fucking ugly f...,0.999,0.680,0.988,0.329,0.929,0.514,0.740
69493,== Go Fuck yourself == \n\n motherfucker mot...,0.999,0.723,0.991,0.265,0.940,0.509,0.738
8441,is a fucking dirty cunt bag. \n\n is a stupid...,1.000,0.760,0.992,0.230,0.956,0.491,0.738
91086,kkk mother fucker fuck niggers the r dirty lit...,0.999,0.688,0.988,0.283,0.928,0.535,0.737


## Exercise

Classify these reviews. 

In [ ]:
try:
    from datasets import load_dataset
except ImportError:
    %pip install datasets
    from datasets import load_dataset

yelp = load_dataset("yelp_review_full")

train_data = yelp["train"]
test_data = yelp["test"]

print(f"Train samples: {len(train_data)}")
print(f"Test samples:  {len(test_data)}")
print(f"\nExample Record: \nlabel: {train_data[0]['label']} \nText: {train_data[0]['text']}...")

README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train samples: 650000
Test samples:  50000

Example Record: 
label: 4 
Text: dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank....
